 # Customer Churn Prediction Project



 **Churn prediction** involves detecting which customers are likely to leave a service or to cancel a subscription. It is a critical metric for businesses to maintain profitability and improve customer satisfaction.



 In this project, we will cover:

 1. Data Cleaning & Preparation

 2. Exploratory Data Analysis (EDA)

 3. Feature Importance (Risk Ratios & Mutual Information)

 4. One-Hot Encoding via `DictVectorizer`

 5. Training and evaluating a Logistic Regression model

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


 ## 1. Data Loading

 We begin by reading the raw dataset into a Pandas DataFrame and inspecting the first few rows.

In [ ]:
df = pd.read_csv("data.csv")
df.head()


 When a dataset has a large number of columns, the standard `.head()` output can truncate columns or be difficult to read.



 We can **transpose** (`.T`) the head of the DataFrame to display the columns as rows, making it much easier to inspect all features at once.

In [ ]:
df.head().T


 ## 2. Data Cleaning

 To maintain a clean and standardized dataset, we will:

 1. Convert all column names to lowercase.

 2. Replace any spaces in column names with underscores.

 3. Apply this exact same string normalization to the values inside all categorical (string/`object`) columns.

In [ ]:
df.columns = df.columns.str.lower().str.replace(pat= " ", repl= "_")
categorical_columns = list(df.dtypes[df.dtypes=="object"].index)

for category in categorical_columns:
    df[category] = df[category].str.lower().str.replace(pat=" ", repl="_")

df.head().T


 Looking closely at the data types, we notice a discrepancy: `totalcharges` is conceptually a numeric value, but its Pandas data type is `object` (string).

In [ ]:
df.dtypes


 ### Fixing Incorrect Data Types

 We attempt to convert the `totalcharges` column into numerical values using the `pd.to_numeric` method.

In [ ]:
pd.to_numeric(df.totalcharges)


 We immediately encounter a `ValueError`.



 **Why did this happen?**

 In the initial dataset, missing values in `totalcharges` were filled with empty spaces. During our data cleaning step, we replaced all spaces with underscores (`_`). Pandas cannot convert an underscore into a number.



 To fix this, we use the `errors='coerce'` argument. This forces Pandas to replace any non-convertible string with a `NaN` (Not a Number) value.

In [ ]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges[df.totalcharges.isnull()]


 Because there are only a handful of missing values relative to the entire dataset, we can safely replace these `NaN` entries with `0`.

In [ ]:
df.totalcharges = df.totalcharges.fillna(0)


 ### Binarizing the Target Variable

 For binary classification tasks like Churn Prediction, machine learning models require numerical targets. We convert the `churn` column from `yes`/`no` string values into `1` and `0`.

In [ ]:
df.churn = df['churn'].apply(lambda x: 1 if x == 'yes' else 0)


 ## 3. Setting Up the Validation Framework

 Previously, you might have split datasets manually using NumPy indices. Going forward, we will utilize the robust `train_test_split` function provided by the **Scikit-Learn** library.

In [ ]:
from sklearn.model_selection import train_test_split


 We will split our dataset into three subsets:

 * **Training Data**: 60%

 * **Validation Data**: 20%

 * **Test Data**: 20%



 *Note: We set `random_state=1` (a random seed) to ensure our splits are reproducible.*

In [ ]:
df_train_plus_val, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_train_plus_val, test_size=0.25, random_state=1)


 Let's verify the length of our subsets to ensure the math checks out.

In [ ]:
len(df_train), len(df_val), len(df_test)


 Finally, we reset the indices of our new DataFrames, extract the target variable (`churn`) into separate NumPy arrays, and delete the target column from the feature DataFrames to prevent data leakage.

In [ ]:
df_train.reset_index(drop=True, inplace=True)
df_val.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']


 ## 4. Exploratory Data Analysis (EDA)

 We will conduct our Exploratory Data Analysis using the combined `train` + `val` dataset to get a comprehensive view of the data without touching the hold-out test set.

In [ ]:
df_train_plus_val.reset_index(drop=True, inplace=True)


 Let's start by verifying if there are any remaining missing values in our dataset.

In [ ]:
df_train_plus_val.isnull().sum()


 There are zero missing values, which means no further imputation is required.



 Next, we look at the raw distribution of our target variable (`churn`).

In [ ]:
df_train_plus_val.churn.value_counts()


 To see the exact proportion (percentage) of churned vs. non-churned customers, we can pass `normalize=True` to the `value_counts` method.

In [ ]:
df_train_plus_val.churn.value_counts(normalize=True)


 We observe an exact churn rate of **~26.3%**.



 *(Tip: Because our target variable is strictly composed of 1s and 0s, simply calculating the `mean()` of the column also yields this exact churn rate).*



 Let's identify the feature types available to us.

In [ ]:
df_train_plus_val.dtypes


 We will organize our features into lists:

 * **Numerical variables**: `tenure` (how long the customer has stayed), `monthlycharges`, and `totalcharges`.

 * **Categorical variables**: demographics and service subscriptions.

In [ ]:
numerical = ["tenure", "monthlycharges", "totalcharges"] 
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents','phoneservice','multiplelines',
        'internetservice','onlinesecurity','onlinebackup','deviceprotection', 'techsupport',
        'streamingtv','streamingmovies', 'contract', 'paperlessbilling','paymentmethod']


 Let's check how many unique values exist within each categorical variable.

In [ ]:
df_train_plus_val[categorical].nunique()


 ## 5. Feature Importance (Categorical)

 To determine how important a categorical variable is, we can compare the churn rates of its specific subgroups.



 First, let's look at **Gender**:

In [ ]:
churn_female = df_train_plus_val[df_train_plus_val.gender == "female"].churn.mean()
churn_female


In [ ]:
churn_male = df_train_plus_val[df_train_plus_val.gender == "male"].churn.mean()
churn_male


 There is virtually no difference between the churn rate of females and males. Therefore, `gender` is not a very useful variable for predicting churn.



 Next, let's look at the **Partner** variable (whether the customer has a partner or not).

In [ ]:
churn_no_partner = df_train_plus_val[df_train_plus_val.partner == "no"].churn.mean()
churn_no_partner


In [ ]:
churn_partner = df_train_plus_val[df_train_plus_val.partner == "yes"].churn.mean()
churn_partner


 ### Risk Ratio

 We see a significant difference! Customers without a partner churn at a much higher rate. This makes `partner` a strong predictive feature.



 Another statistical way to assess feature importance is by calculating the **Risk Rate**:



 $$ \text{Risk Rate} = \frac{\text{Group Churn Rate}}{\text{Global Churn Rate}} $$



 * **Risk > 1**: The group is more likely to churn (negative impact).

 * **Risk $\approx$ 1**: The group holds the same risk as the general population (low impact).

 * **Risk < 1**: The group is less likely to churn (positive impact).

In [ ]:
global_churn = df_train_plus_val.churn.mean()
churn_no_partner / global_churn


 Let's write a loop to generate a comprehensive report of differences and risk rates for all categorical variables in our dataset.

In [ ]:
for category in categorical:
    print(category)
    df_group = df_train_plus_val.groupby(category).churn.agg(['mean', 'count'])
    df_group["diff"] = global_churn - df_group["mean"]
    df_group["risk"] = df_group["mean"] / global_churn
    display(df_group)
    print()
    print()
    print()


 ### Mutual Information

 **Mutual Information** is a concept from Information Theory that tells us how much information we learn about one variable if we observe the value of another. Higher values indicate higher dependency.



 Scikit-Learn implements this for us via `mutual_info_score`. Let's test it on a few variables.

In [ ]:
from sklearn.metrics import mutual_info_score
mutual_info_score(df_train_plus_val.contract, df_train_plus_val.churn)


In [ ]:
mutual_info_score(df_train_plus_val.gender, df_train_plus_val.churn)


In [ ]:
mutual_info_score(df_train_plus_val.partner, df_train_plus_val.churn) 


 To systematically view mutual information scores, we define a helper function and map it across all categorical features, sorting the output in descending order.

In [ ]:
def mutual_info_score_churn(series):
    return mutual_info_score(series, df_train_plus_val.churn) 


In [ ]:
mutual_information = df_train_plus_val[categorical].apply(mutual_info_score_churn)
mutual_information.sort_values(ascending=False)


 ## 6. Feature Importance (Numerical)

 For numerical variables, we evaluate importance by looking at their **Correlation** with the target variable. We use the `.corrwith()` method.

In [ ]:
df_train_plus_val[numerical].corrwith(df_train_plus_val.churn)


 ## 7. One-Hot Encoding

 Machine Learning models only understand numbers. To feed categorical data into a model, we must convert each distinct category value into a binary feature column (`0` or `1`). This process is known as **One-Hot Encoding**.



 Instead of doing this manually, we use Scikit-Learn's `DictVectorizer`.

In [ ]:
from sklearn.feature_extraction import DictVectorizer


 We convert our Pandas DataFrame into a list of dictionaries (where each dictionary is a row). The `DictVectorizer` will automatically recognize strings and convert them into one-hot encoded binary matrices, while leaving numerical values untouched.

In [ ]:
train_dicts = df_train[categorical + numerical].to_dict(orient="records")
dv = DictVectorizer(sparse=False)
dv.fit(train_dicts)


In [ ]:
dv.transform(train_dicts)


 We can check the explicit names of the new columns generated by the vectorizer.

In [ ]:
dv.get_feature_names_out()


 Let's process our training set into a fully prepared feature matrix (`X_train`).

In [ ]:
X_train = dv.fit_transform(train_dicts)


 We repeat the exact same transformation process for our validation dataset.



 *(Note: We use `.transform()` here, not `.fit_transform()`, because we want the vectorizer to apply the exact same column structure it learned from the training data).*

In [ ]:
val_dicts = df_val[categorical + numerical].to_dict(orient="records")
X_val = dv.transform(val_dicts)


 ## 8. Logistic Regression Mechanics

 Linear regression outputs unbounded continuous numbers. To predict binary categories (like Churn = 1 or 0), we need to output probabilities between `0.0` and `1.0`.



 We achieve this by wrapping the linear equation inside a **Sigmoid** function:

 $$ S(z) = \frac{1}{1 + e^{-z}} $$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-5, 5, 51)
output = sigmoid(z)
plt.plot(z, sigmoid(z))


 Conceptually, the Logistic Regression prediction algorithm calculates a linear score based on weights (`w`) and bias (`w0`), and then passes that score into the sigmoid function.

In [ ]:
def logestic_regression(xi):
    score = w0 

    for j in range(len(w)):
        score += xi[j] * w[j]

    result = sigmoid(score)
    return result


 ## 9. Training the Logistic Regression Model

 Let's move from our conceptual understanding to building the actual model using Scikit-Learn.

In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)


 Once trained, we can inspect the array of weights (coefficients) assigned to our features:

In [ ]:
model.coef_[0].round(3)


 And we can view the model's bias (intercept or $w_0$):

In [ ]:
model.intercept_[0].round(3)


 ### Generating Predictions

 We can extract two types of predictions from our model:

 1. **Hard Predictions**: `model.predict()` evaluates probabilities and strictly returns `0` or `1` (using a default threshold of 0.5).

 2. **Soft Predictions (Probabilities)**: `model.predict_proba()` returns the exact probability confidence.

In [ ]:
model.predict(X_train)


 `predict_proba` returns a 2D array.

 * The **first column** is the probability of the negative class (Not Churning).

 * The **second column** is the probability of the positive class (Churning).

In [ ]:
model.predict_proba(X_train)


 Because we are specifically interested in the probability of a customer churning, we will slice the array to keep only the second column.

In [ ]:
y_pred = model.predict_proba(X_train)[:, 1]
y_pred


In [ ]:
y_pred_val = model.predict_proba(X_val)[:, 1]
y_pred_val


 ## 10. Model Evaluation

 We convert our soft probabilities into hard binary decisions using a standard `>= 0.5` threshold. Then, we calculate the model's **Accuracy** by finding the mean of matching predictions against actual targets.

In [ ]:
churn_decision = (y_pred_val >= 0.5).astype(int)
accuracy = (churn_decision == y_val).mean()
accuracy


 Let's build a clear DataFrame mapping out our probabilities, predictions, and the ground-truth actual values side-by-side.

In [ ]:
df_prediction = pd.DataFrame()
df_prediction["probability"] = y_pred_val
df_prediction["prediction"] = churn_decision
df_prediction["actual_values"] = y_val


In [ ]:
df_prediction["correct"] = df_prediction.prediction == df_prediction.actual_values


In [ ]:
df_prediction


 Calculating the mean of the `correct` boolean column yields our overall accuracy, verifying our previous calculation.

In [ ]:
df_prediction.correct.mean()


 ## 11. Model Interpretation

 To understand our model's logic, we can zip the feature names generated by our `DictVectorizer` together with the trained coefficients.



 Positive weights increase churn likelihood, while negative weights decrease it.

In [ ]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))


 ### Training a Smaller Interpretable Model

 Reviewing dozens of features is difficult. Let's train a lightweight model using only three high-impact features (`contract`, `tenure`, `monthlycharges`) so we can easily understand the math.

In [ ]:
small = ['contract', 'tenure', 'monthlycharges']
dicts_train_small = df_train[small].to_dict(orient="records")
dicts_val_small = df_val[small].to_dict(orient="records")


In [ ]:
dv_small = DictVectorizer(sparse=False)
dv_small.fit(dicts_train_small)


In [ ]:
X_train_small = dv_small.transform(dicts_train_small)


In [ ]:
model_small = LogisticRegression()
model_small.fit(X_train_small, y_train)


 Checking the bias/intercept ($w_0$) of the small model:

In [ ]:
w0 = model_small.intercept_[0]
w0


 Checking the coefficients ($w$) of the small model:

In [ ]:
w = model_small.coef_[0].round(3)
w


 Zipping them together to see exactly how these three specific features drive the model's logic.

In [ ]:
dict(zip(dv_small.get_feature_names_out(), w.round(3)))


 ## 12. Final Deployment on Test Data

 Now that we are satisfied with our hyper-parameters and feature selection, we merge the `train` and `val` datasets together to train a final robust model using maximum data volume.



 Finally, we evaluate this production-ready model on our untouched `test` dataset.

In [ ]:
dicts_train_plus_val = df_train_plus_val[categorical + numerical].to_dict(orient="records")
dv = DictVectorizer(sparse=False)
X_train_plus_val = dv.fit_transform(dicts_train_plus_val)
y_train_plus_val = df_train_plus_val.churn.values
model = LogisticRegression(max_iter=5000)
model.fit(X_train_plus_val, y_train_plus_val)


In [ ]:
dicts_test = df_test[categorical + numerical].to_dict(orient="records")
X_test = dv.transform(dicts_test)
y_pred_test = model.predict_proba(X_test)[:, 1]
churn_decision_test = (y_pred_test >= 0.5).astype(int)
(churn_decision_test == y_test).mean()


 ### Single Customer Prediction

 Simulating an API endpoint in production: we take the dictionary record of a single random customer from our test set, pass it through our vectorizer, and generate a probability score for their likelihood to churn.

In [ ]:
random_customer = dicts_test[10]


In [ ]:
X_customer = dv.transform([random_customer])


In [ ]:
model.predict_proba(X_customer)[0][1]


 Checking against the ground truth for this specific customer to see if the prediction was accurate.

In [ ]:
y_test[10]